In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

params = quote_plus(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=localhost,1433;"
    "DATABASE=master;"
    "UID=sa;"
    "PWD=YourStrong!Passw0rd;"
    "TrustServerCertificate=yes;"
    "Encrypt=no;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")
print("Engine created.")

In [ ]:
from pathlib import Path

csv_dir = Path(".")  # fraud/ folder — adjust if running from a different cwd
exclude = {"time_series_stats"}

for csv_file in sorted(csv_dir.glob("*.csv")):
    table_name = csv_file.stem  # filename without extension

    if table_name in exclude:
        print(f"Skipping {csv_file.name}")
        continue

    df = pd.read_csv(csv_file)

    # Parse any column named 'timestamp', 'hour', or 'date' as datetime
    for col in df.columns:
        if col.lower() in ("timestamp", "hour", "date"):
            df[col] = pd.to_datetime(df[col], errors="coerce")

    df.to_sql(
        name=table_name,
        con=engine,
        schema="dbo",
        if_exists="replace",
        index=False,
    )
    print(f"Loaded {csv_file.name} → dbo.{table_name} ({len(df)} rows, {len(df.columns)} cols)")

print("\nAll CSVs loaded.")